# 🏪 ERP · SEMANA 1: DEL MICRONEGOCIO AL PROBLEMA
**CETIS 125 | Módulo IV: Desarrolla aplicaciones web en un sistema de información | Submódulo 1: Construye aplicaciones web**

**Objetivo de la semana:** recopilar el requerimiento del software con técnicas de indagación, redactar historias de usuario con criterios de aceptación y definir el alcance mínimo (MVP) del ERP.

**Producto de la semana:** 📄 *Documento de requerimientos + historias de usuario* (Google Docs): **10 %** de la calificación.

| Sesión | Tema | Reto | Nivel |
|---|---|---|---|
| 1 | Calentamiento: Colab y Python en 8 mini-retos | R1.1 | Base |
| 2 | Arranque del proyecto: pareja, micronegocio y carpeta del ERP | R1.1 | ERP |
| 3 | Empatía con el problema: el mapa de empatía | R1.2 | Base + ERP |
| 4 | El enunciado del problema y la guía de entrevista | R1.2 · R1.3 | ERP + Base |
| 5 | Indagación: encuesta en Forms y clasificación de requerimientos | R1.3 | Base |
| 6 | Los requerimientos de TU negocio, bien escritos | R1.3 | ERP |
| 7 | Historias de usuario, alcance MVP y documento de requerimientos | R1.4 | Base + ERP · **Entrega 10 %** |

### 🔁 Así trabajamos cada sesión (50 min)
| | Momento | Qué haces |
|---|---|---|
| 🔮 | **Predice** (5 min) | Lees un fragmento de código y escribes qué crees que imprimirá **antes** de ejecutarlo |
| 👩‍🏫 | **Mini demo** (10 min) | El docente muestra el concepto del día |
| 🧩 | **Reto en pareja** (25 min) | **Piloto** escribe y **copiloto** revisa y pregunta. Cambian de rol a mitad del reto |
| ✅ | **Autoverificador** (5 min) | Una celda revisa tu trabajo y te da un porcentaje |
| 🎟️ | **Ticket de salida** (5 min) | Anotas en tu bitácora qué aprendiste y qué te bloqueó |

**Tres niveles de reto:** 🟢 **Base** (todas las parejas) · 🔵 **ERP** (lo aplicas a tu negocio) · ⭐ **Bonus** (opcional, si terminas antes).

> 💡 **Antes de empezar:** ejecuta la **CELDA 0**. Todo lo que hagas se guarda en la carpeta de tu proyecto en Google Drive.

---
### 📌 [CELDA 0] Preparar el entorno
Monta Google Drive y carga las herramientas del curso:

| Función | Para qué sirve |
|---|---|
| `predecir(codigo, mi_prediccion)` | Compara tu predicción con lo que realmente imprime el código |
| `autoverificar(titulo, pruebas)` | Revisa tu reto y te da un porcentaje |
| `guardar_json(nombre, datos)` / `leer_json(nombre)` | Guarda y lee tu trabajo en la carpeta del proyecto |
| `ticket_salida(sesion, aprendi, me_bloqueo)` | Registra tu ticket de salida en la bitácora |
| `registrar_ia(pregunta, respuesta, que_acepte, como_verifique)` | Registra cada uso de Gemini en la bitácora de IA |
| `mostrar(html)` · `con_css(html, css)` · `comparar_pantallas(html)` | Muestran páginas HTML dentro de Colab |
| `radiografia(html)` · `revisar_html(html)` | Revisan la semántica y la accesibilidad de una página |
| `arbol()` | Muestra las carpetas y archivos de tu proyecto |

In [2]:
#@title ▶️ Ejecuta esta celda SIEMPRE primero (doble clic para ver el código) { display-mode: "form" }
import os
import re
import html as html_lib
from html.parser import HTMLParser
from IPython.display import HTML, display

# 1. Montar Google Drive y ubicar la carpeta del proyecto ERP
import io
import csv
import json
import base64
import contextlib
from datetime import datetime

try:
    from google.colab import drive
    print("🔄 Conectando con Google Drive...")
    drive.mount('/content/drive')
    RUTA_DRIVE = '/content/drive/MyDrive'
except ImportError:
    RUTA_DRIVE = os.path.join(os.getcwd(), 'MyDrive')

FICHA = os.path.join(RUTA_DRIVE, 'ERP_ficha_proyecto.json')


def _ruta_del_proyecto():
    # Si ya creaste tu ficha (sesión 2), trabajamos en la carpeta ERP_<nombre de tu ERP>
    if os.path.exists(FICHA):
        with open(FICHA, encoding='utf-8') as f:
            return json.load(f)['carpeta']
    return os.path.join(RUTA_DRIVE, 'ERP_practicas')


RUTA_PROYECTO = _ruta_del_proyecto()
RUTA_SEMANA = RUTA_PROYECTO          # guardar() y leer() trabajan en la carpeta del proyecto
os.makedirs(RUTA_PROYECTO, exist_ok=True)


# 2. Guardar y leer archivos en Drive
def guardar(nombre_archivo, contenido):
    ruta = os.path.join(RUTA_SEMANA, nombre_archivo)
    with open(ruta, 'w', encoding='utf-8') as f:
        f.write(contenido)
    print(f"💾 Guardado en Drive: {ruta}")


def leer(nombre_archivo):
    ruta = os.path.join(RUTA_SEMANA, nombre_archivo)
    with open(ruta, 'r', encoding='utf-8') as f:
        return f.read()


# 3. Mostrar páginas HTML dentro de Colab (cada una en su propio marco)
def _marco(codigo_html, ancho='100%', alto=500):
    return (f'<iframe srcdoc="{html_lib.escape(codigo_html, quote=True)}" '
            f'style="width:{ancho};height:{alto}px;border:1px solid #cbd5e0;'
            f'border-radius:8px;background:#fff"></iframe>')


def mostrar(codigo_html, alto=500):
    display(HTML(_marco(codigo_html, '100%', alto)))


def lado_a_lado(html_a, html_b, titulo_a='A', titulo_b='B', alto=320):
    columnas = ''
    for titulo, pagina in ((titulo_a, html_a), (titulo_b, html_b)):
        columnas += (f'<div style="flex:1;min-width:280px">'
                     f'<div style="font:bold 14px sans-serif;margin-bottom:4px">{titulo}</div>'
                     f'{_marco(pagina, "100%", alto)}</div>')
    display(HTML(f'<div style="display:flex;gap:16px;flex-wrap:wrap">{columnas}</div>'))


def con_css(codigo_html, css):
    # Inserta una hoja de estilos <style> justo antes de </head>
    return codigo_html.replace('</head>', f'<style>\n{css}\n</style>\n</head>', 1)


def comparar_pantallas(codigo_html, anchos=(360, 768, 1280), alto=560):
    marcos = ''
    for ancho in anchos:
        escala = min(1, 380 / ancho)
        icono = '📱' if ancho < 600 else ('📟' if ancho < 1024 else '🖥️')
        marcos += (f'<div style="flex:none">'
                   f'<div style="font:bold 13px sans-serif;margin-bottom:4px">{icono} {ancho}px</div>'
                   f'<div style="width:{ancho * escala:.0f}px;height:{alto}px;overflow:hidden">'
                   f'<div style="width:{ancho}px;transform:scale({escala});transform-origin:0 0">'
                   f'{_marco(codigo_html, f"{ancho}px", round(alto / escala))}</div></div></div>')
    display(HTML(f'<div style="display:flex;gap:16px;overflow-x:auto;padding:8px 0">{marcos}</div>'))


# 4. Analizador de HTML: radiografía semántica y revisor automático
REGIONES = {
    'header': 'banner (encabezado)',
    'nav': 'navigation (navegación)',
    'main': 'main (contenido principal)',
    'section': 'region (sección temática)',
    'article': 'article (contenido independiente)',
    'aside': 'complementary (complementario)',
    'footer': 'contentinfo (pie de página)',
    'form': 'form (formulario)',
}


class _Analizador(HTMLParser):
    def __init__(self):
        super().__init__()
        self.etiquetas = []
        self.doctype = ''
        self.titulo = ''
        self.niveles = []
        self.labels_for = set()
        self.controles = []
        self.botones_vacios = 0
        self.mapa = []
        self._prof = 0
        self._en_title = False
        self._labels_abiertos = 0
        self._boton = None
        self._encabezado = None

    def handle_decl(self, decl):
        self.doctype = decl

    def handle_starttag(self, tag, attrs):
        a = {k: (v or '') for k, v in attrs}
        self.etiquetas.append((tag, a))
        if tag == 'title':
            self._en_title = True
        if tag == 'label':
            self._labels_abiertos += 1
            if a.get('for'):
                self.labels_for.add(a['for'])
        if tag in ('input', 'select', 'textarea'):
            self.controles.append((tag, a, self._labels_abiertos > 0))
        if tag == 'button':
            self._boton = 'x' if a.get('aria-label') else ''
        if tag in REGIONES:
            self.mapa.append((self._prof, f"🏷️  <{tag}>  →  {REGIONES[tag]}"))
            self._prof += 1
        if re.fullmatch(r'h[1-6]', tag):
            self._encabezado = [int(tag[1]), '']
            self.niveles.append(int(tag[1]))

    def handle_endtag(self, tag):
        if tag == 'title':
            self._en_title = False
        if tag == 'label':
            self._labels_abiertos = max(0, self._labels_abiertos - 1)
        if tag == 'button' and self._boton is not None:
            if not self._boton.strip():
                self.botones_vacios += 1
            self._boton = None
        if tag in REGIONES:
            self._prof = max(0, self._prof - 1)
        if self._encabezado and tag == f'h{self._encabezado[0]}':
            nivel, texto = self._encabezado
            self.mapa.append((self._prof, f"📑 h{nivel}: {' '.join(texto.split())}"))
            self._encabezado = None

    def handle_data(self, data):
        if self._en_title:
            self.titulo += data
        if self._boton is not None:
            self._boton += data
        if self._encabezado:
            self._encabezado[1] += data


def _analizar(codigo_html):
    analizador = _Analizador()
    analizador.feed(codigo_html)
    return analizador


def radiografia(codigo_html, nombre='la página'):
    a = _analizar(codigo_html)
    print(f"🔎 Radiografía de {nombre}: lo que entiende un navegador, Google o un lector de pantalla")
    print("-" * 75)
    if not a.mapa:
        print("⚠️  No encontré regiones ni encabezados.")
        print("    Para un lector de pantalla, esta página es un solo bloque de texto sin estructura.")
    for profundidad, linea in a.mapa:
        print("    " * profundidad + linea)
    etiquetas = [t for t, _ in a.etiquetas]
    semanticas = sum(etiquetas.count(t) for t in REGIONES)
    print("-" * 75)
    print(f"📊 Etiquetas semánticas: {semanticas}   |   <div>: {etiquetas.count('div')}")


def revisar_html(codigo_html, nombre='la maqueta'):
    a = _analizar(codigo_html)
    etiquetas = [t for t, _ in a.etiquetas]
    de_tipo = lambda tag: [attrs for t, attrs in a.etiquetas if t == tag]
    metas = de_tipo('meta')
    raiz = de_tipo('html')
    semanticas = sum(etiquetas.count(t) for t in REGIONES)

    sin_label = []
    for tag, attrs, dentro_de_label in a.controles:
        if attrs.get('type') in ('hidden', 'submit', 'button', 'reset', 'image'):
            continue
        tiene = (dentro_de_label or attrs.get('id') in a.labels_for
                 or attrs.get('aria-label') or attrs.get('aria-labelledby'))
        if not tiene:
            sin_label.append(attrs.get('id') or attrs.get('name') or attrs.get('placeholder') or tag)

    botones_falsos = [x.get('class') for t, x in a.etiquetas
                      if t in ('div', 'span') and re.search(r'bot[oó]n|btn', x.get('class', ''))]
    saltos = [f"h{p}→h{n}" for p, n in zip(a.niveles, a.niveles[1:]) if n > p + 1]
    imgs_sin_alt = [x.get('src', '?') for x in de_tipo('img') if 'alt' not in x]

    pruebas = [
        (a.doctype.lower().startswith('doctype html'),
         'Declara <!DOCTYPE html>', 'Escribe <!DOCTYPE html> en la primera línea.'),
        (bool(raiz) and bool(raiz[0].get('lang')),
         'La etiqueta <html> indica el idioma', 'Usa <html lang="es"> para que el lector de pantalla pronuncie en español.'),
        (any('charset' in m for m in metas),
         'Declara la codificación (meta charset)', 'Agrega <meta charset="UTF-8"> para que se vean bien los acentos y la ñ.'),
        (any(m.get('name') == 'viewport' for m in metas),
         'Tiene meta viewport (lista para celular)', 'Agrega <meta name="viewport" content="width=device-width, initial-scale=1.0">.'),
        (bool(a.titulo.strip()),
         'Tiene un <title> descriptivo', 'Escribe un <title> con el nombre de tu app.'),
        (etiquetas.count('h1') == 1,
         'Usa exactamente un <h1>', f'Hay {etiquetas.count("h1")} <h1>; deja solo uno, con el nombre de la app o de la pantalla.'),
        (bool(a.niveles) and not saltos,
         'Los encabezados siguen un orden (sin saltos)', 'Ordena los títulos h1 → h2 → h3. ' + (f'Saltos: {", ".join(saltos)}' if saltos else 'No hay encabezados.')),
        ('header' in etiquetas,
         'Tiene <header>', 'Envuelve el logo y el título de tu app en <header>.'),
        ('nav' in etiquetas,
         'Tiene <nav>', 'Pon el menú principal dentro de <nav>.'),
        (etiquetas.count('main') == 1,
         'Tiene un solo <main>', f'Hay {etiquetas.count("main")} <main>; el contenido principal va en un único <main>.'),
        ('footer' in etiquetas,
         'Tiene <footer>', 'Agrega un <footer> con los créditos de tu proyecto.'),
        (not imgs_sin_alt,
         'Todas las imágenes tienen texto alternativo (alt)', f'Imágenes sin alt: {", ".join(imgs_sin_alt)}'),
        (not sin_label,
         'Cada campo del formulario tiene <label>', f'Campos sin etiqueta: {", ".join(sin_label)}. El placeholder NO sustituye a <label>.'),
        (a.botones_vacios == 0,
         'Todos los botones tienen texto', 'Escribe un texto dentro de cada <button>, o agrega aria-label.'),
        (not botones_falsos,
         'Los botones son <button> (no <div> disfrazados)', 'Un <div class="boton"> no se puede activar con teclado ni lo anuncia el lector de pantalla. Usa <button>.'),
        ('TODO' not in codigo_html,
         'No quedan TODO pendientes', f'Quedan {codigo_html.count("TODO")} TODO por completar con los datos de tu proyecto.'),
        ('table' not in etiquetas or 'th' in etiquetas,
         'Las tablas tienen encabezados <th>', 'Usa <th scope="col"> en la primera fila de tus tablas.'),
        (etiquetas.count('div') <= semanticas,
         'Sin "divitis" (más etiquetas semánticas que <div>)', f'Hay {etiquetas.count("div")} <div> y {semanticas} etiquetas semánticas. ¿Cuáles <div> podrían ser header, nav, section…?'),
    ]

    aprobadas = sum(ok for ok, _, _ in pruebas)
    print(f"🧪 Revisión automática de {nombre}")
    print("=" * 75)
    for ok, criterio, consejo in pruebas:
        print(f"✅ {criterio}" if ok else f"❌ {criterio}\n      💡 {consejo}")
    print("=" * 75)
    porcentaje = round(100 * aprobadas / len(pruebas))
    medalla = '🏆' if porcentaje == 100 else ('👍' if porcentaje >= 80 else '🔧')
    print(f"{medalla} Resultado: {aprobadas}/{len(pruebas)} criterios ({porcentaje} %)")
    return porcentaje


# 5. Predicciones: ¿qué imprimirá este código?
def predecir(codigo, mi_prediccion):
    if not str(mi_prediccion).strip():
        print("✏️ Primero escribe tu predicción en MI_PREDICCION y vuelve a ejecutar.")
        return
    salida = io.StringIO()
    with contextlib.redirect_stdout(salida):
        exec(codigo, {})
    real = salida.getvalue().strip()
    if ' '.join(str(mi_prediccion).split()) == ' '.join(real.split()):
        print(f"✅ ¡Acertaste! El código imprime: {real}")
    else:
        print(f"🤔 Tu predicción:  {mi_prediccion}")
        print(f"   Resultado real: {real}")
        print("   💬 Coméntalo con tu pareja: ¿en qué línea cambió lo que esperabas?")


# 6. Autoverificador: cada prueba es (descripción, función que devuelve True/False, pista)
def autoverificar(titulo, pruebas, meta=None):
    aprobadas = 0
    print(f"✅ Autoverificador · {titulo}")
    print("=" * 75)
    for descripcion, prueba, pista in pruebas:
        try:
            ok = bool(prueba())
            error = ''
        except Exception as e:
            ok = False
            error = f" ({type(e).__name__}: {e})"
        aprobadas += ok
        print(f"✅ {descripcion}" if ok else f"❌ {descripcion}{error}\n      💡 {pista}")
    print("=" * 75)
    porcentaje = round(100 * aprobadas / len(pruebas))
    meta_txt = f" · meta: {meta}" if meta else ''
    medalla = '🏆' if porcentaje == 100 else ('👍' if porcentaje >= 75 else '🔧')
    print(f"{medalla} Resultado: {aprobadas}/{len(pruebas)} ({porcentaje} %){meta_txt}")
    return porcentaje


# 7. Guardar y leer el trabajo del proyecto (JSON en la carpeta docs/)
def guardar_json(nombre, datos):
    ruta = os.path.join(RUTA_PROYECTO, 'docs', nombre)
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    print(f"💾 Guardado en tu proyecto: docs/{nombre}")


def leer_json(nombre, por_defecto=None):
    ruta = os.path.join(RUTA_PROYECTO, 'docs', nombre)
    if not os.path.exists(ruta):
        return por_defecto
    with open(ruta, encoding='utf-8') as f:
        return json.load(f)


# 8. Bitácoras: tickets de salida y uso de IA
def _anotar(archivo, encabezado, fila):
    ruta = os.path.join(RUTA_PROYECTO, 'bitacora', archivo)
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    nuevo = not os.path.exists(ruta)
    with open(ruta, 'a', encoding='utf-8', newline='') as f:
        escritor = csv.writer(f)
        if nuevo:
            escritor.writerow(encabezado)
        escritor.writerow(fila)


def ticket_salida(sesion, aprendi, me_bloqueo):
    if not aprendi.strip():
        print("✏️ Escribe qué aprendiste hoy antes de registrar tu ticket.")
        return
    _anotar('tickets_salida.csv', ['fecha', 'sesion', 'aprendi', 'me_bloqueo'],
            [f"{datetime.now():%Y-%m-%d %H:%M}", sesion, aprendi.strip(), me_bloqueo.strip()])
    print(f"🎟️ Ticket de la sesión {sesion} registrado en bitacora/tickets_salida.csv")


def registrar_ia(pregunta, respuesta, que_acepte, como_verifique):
    _anotar('bitacora_ia.csv', ['fecha', 'pregunta', 'respuesta', 'que_acepte', 'como_verifique'],
            [f"{datetime.now():%Y-%m-%d %H:%M}", pregunta, respuesta, que_acepte, como_verifique])
    print("🤖 Uso de IA registrado en bitacora/bitacora_ia.csv")


# 9. Ver el árbol de carpetas del proyecto
def arbol(ruta=None, prefijo=''):
    ruta = ruta or RUTA_PROYECTO
    if not prefijo:
        print(f"📁 {os.path.basename(ruta)}/")
    elementos = sorted(os.listdir(ruta))
    for i, nombre in enumerate(elementos):
        ultimo = i == len(elementos) - 1
        completo = os.path.join(ruta, nombre)
        print(f"{prefijo}{'└── ' if ultimo else '├── '}{'📁 ' if os.path.isdir(completo) else '📄 '}{nombre}")
        if os.path.isdir(completo):
            arbol(completo, prefijo + ('    ' if ultimo else '│   '))


print("=" * 75)
print(f"📁 Carpeta del proyecto:\n{RUTA_PROYECTO}")
print("🧰 Herramientas cargadas: predecir, autoverificar, guardar_json, leer_json, ticket_salida,")
print("   registrar_ia, arbol, mostrar, con_css, comparar_pantallas, radiografia, revisar_html")
print("=" * 75)

🔄 Conectando con Google Drive...
Mounted at /content/drive
📁 Carpeta del proyecto:
/content/drive/MyDrive/ERP_practicas
🧰 Herramientas cargadas: predecir, autoverificar, guardar_json, leer_json, ticket_salida,
   registrar_ia, arbol, mostrar, con_css, comparar_pantallas, radiografia, revisar_html


---
# 🟦 SESIÓN 1 · Calentamiento: Colab y Python en 8 mini-retos
**Reto R1.1 🟢 Base** · Recuperar lo que ya sabes de Python (variables, listas, diccionarios, funciones y ciclos) con ejemplos de un negocio.

### 🔮 Predice
Sin ejecutar: ¿qué imprime este código? Escribe tu predicción en el formulario y luego ejecuta.
```python
precios = [38, 7, 65]
precios.append(14.5)
print(len(precios), precios[-1])
```

In [6]:
MI_PREDICCION = "4 65"  #@param {type:"string"}
predecir("""
precios = [38, 7, 65]
precios.append(14.5)
print(len(precios), precios[2])
""", MI_PREDICCION)

✅ ¡Acertaste! El código imprime: 4 65


### 🧩 Reto R1.1 · 8 mini-retos de un negocio
Completa cada función donde dice `✏️ TODO`. Ejecuta la celda y después el autoverificador. **Meta: 6 de 8.**

In [ ]:
# 1 · VARIABLES Y F-STRINGS: saludo("Ana") → "Hola, Ana. Bienvenido al ERP."
def saludo(nombre):



# 2 · LISTAS: suma de una lista de precios. total_lista([10, 20.5, 30]) → 60.5
def total_lista(precios):
    pass  # ✏️ TODO


# 3 · DICCIONARIOS: precio de un producto; si no existe, devuelve 0
#     precio_de({"cuaderno": 38, "goma": 7}, "goma") → 7
def precio_de(catalogo, producto):
    pass  # ✏️ TODO


# 4 · CONDICIONALES: 10 % de descuento si la compra es de $500 o más
#     aplicar_descuento(600) → 540.0   ·   aplicar_descuento(100) → 100
def aplicar_descuento(total):
    pass  # ✏️ TODO


# 5 · CICLO FOR: ¿cuántos productos tienen existencia menor o igual al mínimo?
#     contar_bajo_stock({"cuaderno": 45, "pluma": 3, "goma": 5}, 5) → 2
def contar_bajo_stock(inventario, minimo):
    pass  # ✏️ TODO


# 6 · LISTA DE DICCIONARIOS: nombres de los productos activos
#     nombres_activos([{"nombre": "Cuaderno", "activo": True}, {"nombre": "Lápiz", "activo": False}]) → ["Cuaderno"]
def nombres_activos(productos):
    pass  # ✏️ TODO


# 7 · CICLO WHILE: ¿cuántas semanas para juntar la meta ahorrando lo mismo cada semana?
#     semanas_para_ahorrar(1000, 150) → 7
def semanas_para_ahorrar(meta, ahorro_semanal):
    pass  # ✏️ TODO


# 8 · CADENAS: clave de producto con las 3 primeras letras en mayúsculas y el número con 3 dígitos
#     clave_producto("cuaderno", 7) → "CUA-007"
def clave_producto(nombre, numero):
    pass  # ✏️ TODO

In [ ]:
autoverificar('8 mini-retos de Python', [
    ('1 · saludo()', lambda: saludo("Ana") == "Hola, Ana. Bienvenido al ERP.", 'Usa un f-string: f"Hola, {nombre}. ..."'),
    ('2 · total_lista()', lambda: total_lista([10, 20.5, 30]) == 60.5 and total_lista([]) == 0, 'Acumula en una variable con un for, o usa sum().'),
    ('3 · precio_de()', lambda: precio_de({"cuaderno": 38, "goma": 7}, "goma") == 7 and precio_de({"goma": 7}, "lápiz") == 0, 'El método .get(clave, valor_por_defecto) evita el KeyError.'),
    ('4 · aplicar_descuento()', lambda: abs(aplicar_descuento(600) - 540) < 0.01 and aplicar_descuento(100) == 100, '¿Usaste >= 500? Multiplica por 0.9 para restar el 10 %.'),
    ('5 · contar_bajo_stock()', lambda: contar_bajo_stock({"cuaderno": 45, "pluma": 3, "goma": 5}, 5) == 2, 'Recorre inventario.values() y cuenta los que sean <= minimo.'),
    ('6 · nombres_activos()', lambda: nombres_activos([{"nombre": "Cuaderno", "activo": True}, {"nombre": "Lápiz", "activo": False}]) == ["Cuaderno"], 'Recorre la lista; cada elemento es un diccionario: p["activo"], p["nombre"].'),
    ('7 · semanas_para_ahorrar()', lambda: semanas_para_ahorrar(1000, 150) == 7 and semanas_para_ahorrar(300, 100) == 3, 'Mientras lo ahorrado sea menor que la meta, suma una semana.'),
    ('8 · clave_producto()', lambda: clave_producto("cuaderno", 7) == "CUA-007" and clave_producto("Goma", 12) == "GOM-012", 'Rebana con [:3], usa .upper() y el formato {numero:03d}.'),
], meta='6 de 8')

### ⭐ Bonus
1. Escribe `ticket_compra(carrito, catalogo)` que reciba `{"cuaderno": 2, "goma": 3}` y el catálogo de precios, e **imprima** cada renglón (producto, cantidad, subtotal) y el total con descuento usando tus funciones 3 y 4.
2. Pregúntale a Gemini cómo mejorar tu función y regístralo con `registrar_ia()`.

In [ ]:
def ticket_compra(carrito, catalogo):
    pass  # ⭐ Bonus

ticket_compra({"cuaderno": 2, "goma": 3}, {"cuaderno": 38, "goma": 7})

### 🎟️ Ticket de salida · Sesión 1

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(1, APRENDI, ME_BLOQUEO)

---
# 🟦 SESIÓN 2 · Arranque del proyecto: pareja, micronegocio y carpeta del ERP
**Reto R1.1 🔵 ERP** · Formar la pareja, recibir el micronegocio, ponerle nombre al ERP y crear la carpeta del proyecto en Drive.

### 🔮 Predice
```python
negocio = "Papelería escolar"
carpeta = "ERP_" + negocio.lower().replace(" ", "_")
print(carpeta)
```

In [ ]:
MI_PREDICCION = ""  #@param {type:"string"}
predecir("""
negocio = "Papelería escolar"
carpeta = "ERP_" + negocio.lower().replace(" ", "_")
print(carpeta)
""", MI_PREDICCION)

### 🏗️ El proyecto integrador: un ERP para un micronegocio
Un **ERP** (*Enterprise Resource Planning*) es un sistema que concentra la operación de un negocio en un solo lugar: lo que vende, lo que tiene en inventario, a quién le vende y cuánto ganó en el día.

Durante el submódulo, cada pareja construye **el ERP web de un micronegocio** en Google Colab (Flask + SQLite), con diseño, CRUD, lógica de negocio, pruebas, manuales y portafolio.

| Parte | Qué incluye |
|---|---|
| **Núcleo común** (todas las parejas) | Productos o servicios · Clientes · Ventas (encabezado y detalle) · Inventario con alerta de stock bajo · Reporte del día |
| **Módulo distintivo** (uno por negocio) | Una función propia del giro que hace único tu sistema |

**Modelo de datos mínimo** (lo implementarás en SQLite más adelante; viene del modelo relacional del módulo III):
```
productos     (id, nombre, precio, stock, stock_minimo, activo)
clientes      (id, nombre, telefono)
ventas        (id, fecha, cliente_id → clientes, total)
detalle_venta (id, venta_id → ventas, producto_id → productos, cantidad, precio_unitario)
+ la tabla de tu módulo distintivo
```

| Micronegocio | Módulo distintivo |
|---|---|
| Papelería escolar | Pedidos de fotocopias e impresiones |
| Cafetería o fonda | Menú del día y pedidos para recoger |
| Tienda de abarrotes | Cuentas fiadas por cliente |
| Barbería o estética | Agenda de citas |
| Taller de reparación de celulares | Órdenes de servicio con estado |
| Pastelería | Pedidos por encargo con fecha de entrega |
| Renta de equipo de sonido | Reservas y calendario de disponibilidad |
| Gimnasio | Membresías y vencimientos |
| Lavandería | Órdenes de lavado y entrega |
| Veterinaria | Expediente de mascotas y citas |

> 📚 **¿Qué dice el programa de estudios?** Esta semana trabajamos la actividad clave **«Diseña aplicaciones web»**: recopilar la información del requerimiento del proyecto usando las TIC para procesar e interpretar los datos, con capacidad de abstracción y una postura crítica. Al terminar el módulo serás capaz de **desarrollar, construir e implementar aplicaciones web** en un sistema de información. *(Programa de estudios de Programación, DGETI.)*

### 👥 Trabajo en pareja: piloto y copiloto
| Rol | Qué hace |
|---|---|
| 🧑‍✈️ **Piloto** | Tiene el teclado y escribe el código |
| 🧭 **Copiloto** | Lee las instrucciones, revisa cada línea, busca en la documentación y pregunta *“¿por qué?”* |

Cambien de rol **a mitad de cada reto**. Los dos deben poder explicar todo el código.

### 🧩 Reto R1.1 ERP · La ficha de tu proyecto
Llena el formulario y ejecuta la celda. Se crea la carpeta **`ERP_<nombre de tu ERP>`** (por ejemplo `ERP_papeletec`) en tu Drive, con las subcarpetas que usarás todo el semestre.

> 📌 **Solo el piloto** crea la carpeta. Después, en Drive, la **comparte con su pareja como Editor**, y su pareja le agrega un **acceso directo** en *Mi unidad* (clic derecho → *Organizar* → *Agregar acceso directo*).

In [ ]:
NEGOCIO = "Papelería escolar"  #@param ["Papelería escolar", "Cafetería o fonda", "Tienda de abarrotes", "Barbería o estética", "Taller de reparación de celulares", "Pastelería", "Renta de equipo de sonido", "Gimnasio", "Lavandería", "Veterinaria", "Otro (lista oficial de casos)"]
NOMBRE_DEL_ERP = ""  #@param {type:"string"}
INTEGRANTES = ""  #@param {type:"string"}
MODULO_DISTINTIVO_SI_ES_OTRO = ""  #@param {type:"string"}

MODULOS = {
    "Papelería escolar": "Pedidos de fotocopias e impresiones",
    "Cafetería o fonda": "Menú del día y pedidos para recoger",
    "Tienda de abarrotes": "Cuentas fiadas por cliente",
    "Barbería o estética": "Agenda de citas",
    "Taller de reparación de celulares": "Órdenes de servicio con estado",
    "Pastelería": "Pedidos por encargo con fecha de entrega",
    "Renta de equipo de sonido": "Reservas y calendario de disponibilidad",
    "Gimnasio": "Membresías y vencimientos",
    "Lavandería": "Órdenes de lavado y entrega",
    "Veterinaria": "Expediente de mascotas y citas",
}


def nombre_de_carpeta(texto):
    limpio = texto.lower()
    for con, sin in zip("áéíóúüñ", "aeiouun"):
        limpio = limpio.replace(con, sin)
    limpio = re.sub(r'[^a-z0-9]+', '_', limpio).strip('_')
    return "ERP_" + limpio


def crear_proyecto():
    global RUTA_PROYECTO, RUTA_SEMANA
    if not NOMBRE_DEL_ERP.strip() or not INTEGRANTES.strip():
        print("✏️ Escribe el nombre del ERP y los integrantes antes de crear el proyecto.")
        return
    modulo = MODULOS.get(NEGOCIO, MODULO_DISTINTIVO_SI_ES_OTRO.strip())
    carpeta = os.path.join(RUTA_DRIVE, nombre_de_carpeta(NOMBRE_DEL_ERP))
    for sub in ('docs', 'bitacora', 'datos', 'templates', 'static'):
        os.makedirs(os.path.join(carpeta, sub), exist_ok=True)

    ficha = {
        "nombre_erp": NOMBRE_DEL_ERP.strip(),
        "negocio": NEGOCIO,
        "modulo_distintivo": modulo,
        "integrantes": [i.strip() for i in re.split(r',| y ', INTEGRANTES) if i.strip()],
        "carpeta": carpeta,
    }
    with open(FICHA, 'w', encoding='utf-8') as f:
        json.dump(ficha, f, ensure_ascii=False, indent=2)

    RUTA_PROYECTO = RUTA_SEMANA = carpeta
    guardar_json('ficha.json', ficha)
    guardar('README.md', f"# {ficha['nombre_erp']}\n\n"
                         f"ERP web para: **{NEGOCIO}**\n\n"
                         f"Módulo distintivo: **{modulo}**\n\n"
                         f"Integrantes: {', '.join(ficha['integrantes'])}\n")
    print()
    arbol()
    return ficha


ficha = crear_proyecto()

In [ ]:
autoverificar('Ficha y carpeta del proyecto', [
    ('Le pusiste nombre al ERP', lambda: bool(leer_json('ficha.json', {}).get('nombre_erp')), 'Escribe NOMBRE_DEL_ERP en el formulario.'),
    ('Registraste a los integrantes', lambda: len(leer_json('ficha.json', {}).get('integrantes', [])) >= 2, 'Separa los nombres con coma: "Ana López, Luis Pérez".'),
    ('Tienes módulo distintivo', lambda: bool(leer_json('ficha.json', {}).get('modulo_distintivo')), 'Si elegiste "Otro", escribe tu módulo distintivo.'),
    ('La carpeta tiene sus 5 subcarpetas', lambda: all(os.path.isdir(os.path.join(RUTA_PROYECTO, s)) for s in ('docs', 'bitacora', 'datos', 'templates', 'static')), 'Ejecuta otra vez la celda de la ficha.'),
    ('Existe el README.md', lambda: os.path.exists(os.path.join(RUTA_PROYECTO, 'README.md')), 'Ejecuta otra vez la celda de la ficha.'),
])

### ⭐ Bonus
Agrega al `README.md` un **“pitch”** de dos líneas: *¿para quién es tu ERP y qué problema le quita?* Usa `guardar('README.md', ...)` con el texto anterior más tu pitch, y comprueba con `arbol()`.

### 🎟️ Ticket de salida · Sesión 2

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(2, APRENDI, ME_BLOQUEO)

---
# 🟦 SESIÓN 3 · Empatía con el problema: el mapa de empatía
**Reto R1.2 🟢 Base + 🔵 ERP** · Comprender a la persona para la que diseñamos antes de pensar en la solución.

### 🔮 Predice
```python
mapa = {"dice": ["No me alcanza el tiempo"], "hace": []}
mapa["hace"].append("Anota las ventas en una libreta")
print(len(mapa), len(mapa["hace"]))
```

In [ ]:
MI_PREDICCION = ""  #@param {type:"string"}
predecir("""
mapa = {"dice": ["No me alcanza el tiempo"], "hace": []}
mapa["hace"].append("Anota las ventas en una libreta")
print(len(mapa), len(mapa["hace"]))
""", MI_PREDICCION)

### 🧠 ¿Qué es un mapa de empatía?
Antes de diseñar un sistema hay que **entender a la persona** que lo usará. El diseño centrado en las personas propone primero **empatizar**: observar, escuchar y comprender a quién se diseña, y solo después definir el problema e idear soluciones. *(Stanford d.school, Design Tools for Creative Thinking.)*

Usaremos un mapa de **4 cuadrantes**:

| Cuadrante | Pregunta guía | Ejemplo (dueño de una tienda) |
|---|---|---|
| 💬 **Dice** | ¿Qué frases dice en voz alta? | *“Al final del día no sé cuánto gané.”* |
| 🛠️ **Hace** | ¿Qué hace hoy, paso a paso? | *Anota los fiados en un cuaderno.* |
| 😣 **Le frustra** | ¿Qué le molesta o le hace perder tiempo o dinero? | *Que se le acaben productos sin darse cuenta.* |
| 🎯 **Necesita** | ¿Qué necesita lograr (no qué app quiere)? | *Saber qué productos debe resurtir.* |

> ⚠️ El mapa describe a la **persona**, no al software. *“Necesita una app”* no es una necesidad; *“necesita saber cuánto vendió sin contar billetes”*, sí.

### 👩‍🏫 Mini demo: el mapa de Don Beto (tienda de abarrotes)

In [ ]:
def tarjeta_empatia(persona, mapa):
    iconos = {"dice": "💬 Dice", "hace": "🛠️ Hace", "le_frustra": "😣 Le frustra", "necesita": "🎯 Necesita",
              "piensa": "💭 Piensa", "siente": "❤️ Siente"}
    cuadrantes = ''
    for clave, frases in mapa.items():
        lista = ''.join(f'<li>{html_lib.escape(f)}</li>' for f in frases)
        cuadrantes += f'<section><h2>{iconos.get(clave, clave)}</h2><ul>{lista}</ul></section>'
    return f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Mapa de empatía · {html_lib.escape(persona)}</title>
<style>
  body {{ font-family: 'Segoe UI', system-ui, sans-serif; margin: 0; padding: 1rem; background: #f0f4f8; color: #2d3748; }}
  header {{ background: #2b6cb0; color: #fff; padding: 1rem; border-radius: 10px 10px 0 0; }}
  header h1 {{ margin: 0; font-size: 1.2rem; }}
  main {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: .75rem;
          background: #fff; padding: 1rem; border-radius: 0 0 10px 10px; }}
  section {{ border: 1px solid #cbd5e0; border-radius: 8px; padding: .75rem; }}
  h2 {{ margin: 0 0 .5rem; font-size: 1rem; }}
  ul {{ margin: 0; padding-left: 1.2rem; }}
</style></head>
<body><header><h1>🧑 {html_lib.escape(persona)}</h1></header><main>{cuadrantes}</main></body></html>"""


mapa_beto = {
    "dice": ["Al final del día no sé cuánto gané.", "Los fiados me tienen loco."],
    "hace": ["Anota los fiados en un cuaderno.", "Cuenta el dinero de la caja al cerrar."],
    "le_frustra": ["Se le acaban productos sin darse cuenta.", "Pierde tiempo buscando quién le debe."],
    "necesita": ["Saber qué productos debe resurtir.", "Cobrar los fiados a tiempo."],
}
mostrar(tarjeta_empatia("Don Beto · dueño de una tienda de abarrotes", mapa_beto), alto=420)

### 🧩 Reto R1.2 Base · El mapa de Doña Lupita (papelería escolar)
Lee las 12 frases de la entrevista y **copia cada una** al cuadrante que le corresponde. Cada frase va en un solo cuadrante.

In [ ]:
FRASES_LUPITA = [
    "En agosto no me doy abasto con las listas escolares.",                   # 0
    "Saca copias y cobra de memoria.",                                         # 1
    "Que los clientes esperen mucho en temporada de regreso a clases.",        # 2
    "Saber qué útiles se venden más para comprar a tiempo.",                   # 3
    "Ya no sé ni cuánto me deben las escuelas.",                               # 4
    "Revisa los anaqueles a ojo para ver qué falta.",                          # 5
    "Que se le pierdan los pedidos de impresión que le mandan por WhatsApp.",  # 6
    "Llevar el control de los pedidos de impresión.",                          # 7
    "Lo que más me deja son las impresiones.",                                 # 8
    "Hace las cuentas del día en una calculadora.",                            # 9
    "Que el proveedor le surta tarde.",                                        # 10
    "Atender más rápido en temporada alta.",                                   # 11
]

mapa_lupita = {
    "dice": [],        # ✏️ copia aquí las frases que corresponden
    "hace": [],
    "le_frustra": [],
    "necesita": [],
}

mostrar(tarjeta_empatia("Doña Lupita · dueña de una papelería escolar", mapa_lupita), alto=420)

NameError: name 'mostrar' is not defined

In [ ]:
_CLAVE = json.loads(base64.b64decode("WyJkaWNlIiwgImhhY2UiLCAibGVfZnJ1c3RyYSIsICJuZWNlc2l0YSIsICJkaWNlIiwgImhhY2UiLCAibGVfZnJ1c3RyYSIsICJuZWNlc2l0YSIsICJkaWNlIiwgImhhY2UiLCAibGVfZnJ1c3RyYSIsICJuZWNlc2l0YSJd"))

def _aciertos():
    return sum(FRASES_LUPITA[i] in mapa_lupita.get(c, []) for i, c in enumerate(_CLAVE))

autoverificar('Mapa de empatía de Doña Lupita', [
    ('Tiene los 4 cuadrantes', lambda: set(mapa_lupita) >= {"dice", "hace", "le_frustra", "necesita"}, 'No borres ninguna clave del diccionario.'),
    ('Usaste las 12 frases', lambda: sum(len(v) for v in mapa_lupita.values()) == 12, 'Cada frase va en un cuadrante, sin repetir.'),
    ('Ninguna frase repetida ni inventada', lambda: sorted(sum(mapa_lupita.values(), [])) == sorted(FRASES_LUPITA), 'Copia las frases exactamente como vienen en FRASES_LUPITA (puedes usar FRASES_LUPITA[0]).'),
    ('Al menos 9 de 12 en el cuadrante correcto', lambda: _aciertos() >= 9, '¿Lo dice ella en primera persona? → dice. ¿Es una acción? → hace. ¿Es un problema? → le frustra. ¿Es una meta? → necesita.'),
], meta='75 % o más')
print(f"🎯 Frases en el cuadrante correcto: {_aciertos()} de 12")

### 🧩 Reto R1.2 ERP · El mapa del dueño de TU negocio
Piensa en el dueño o encargado de tu micronegocio (real, o uno que conozcas). Escribe **al menos 3 frases por cuadrante**. En la sesión 4 prepararán una entrevista para confirmar o corregir este mapa.

In [ ]:
PERSONA = ""  #@param {type:"string"}

mapa_mi_negocio = {
    "dice": ["", "", ""],          # ✏️ frases que dice en voz alta
    "hace": ["", "", ""],          # ✏️ lo que hace hoy, paso a paso
    "le_frustra": ["", "", ""],    # ✏️ lo que le hace perder tiempo o dinero
    "necesita": ["", "", ""],      # ✏️ lo que necesita lograr (no una app)
}

limpio = {k: [f.strip() for f in v if f.strip()] for k, v in mapa_mi_negocio.items()}
guardar_json('mapa_empatia.json', {"persona": PERSONA, "mapa": limpio})
tarjeta = tarjeta_empatia(PERSONA or "Dueño de mi negocio", limpio)
guardar('docs/mapa_empatia.html', tarjeta)
mostrar(tarjeta, alto=440)

In [ ]:
_mapa = lambda: leer_json('mapa_empatia.json', {}).get('mapa', {})
autoverificar('Mapa de empatía de mi negocio', [
    ('Escribiste quién es la persona', lambda: len(leer_json('mapa_empatia.json', {}).get('persona', '')) >= 5, 'Ejemplo: "Don Toño · dueño de la cafetería La Esquina".'),
    ('Los 4 cuadrantes tienen 3 o más frases', lambda: all(len(_mapa().get(c, [])) >= 3 for c in ("dice", "hace", "le_frustra", "necesita")), 'Llena las 3 frases de cada cuadrante.'),
    ('Las necesidades no son una app', lambda: bool(_mapa().get('necesita')) and not any(re.search(r'\b(app|aplicaci[oó]n|sistema|software|programa)\b', f, re.I) for f in _mapa().get('necesita', [])), 'Describe lo que la persona quiere lograr, no la tecnología.'),
    ('La tarjeta quedó guardada en docs/', lambda: os.path.exists(os.path.join(RUTA_PROYECTO, 'docs', 'mapa_empatia.html')), 'Ejecuta la celda anterior.'),
])

### ⭐ Bonus
`tarjeta_empatia()` acepta cuadrantes extra: agrega **`"piensa"`** y **`"siente"`** al mapa de tu negocio y observa cómo la rejilla (Grid `auto-fit`) se reacomoda sola.

### 🎟️ Ticket de salida · Sesión 3

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(3, APRENDI, ME_BLOQUEO)

---
# 🟦 SESIÓN 4 · El enunciado del problema y la guía de entrevista
**Reto R1.2 🔵 ERP · R1.3 🟢 Base** · Formular el problema real del negocio y preparar preguntas de entrevista que no sesguen las respuestas.

### 🔮 Predice
```python
persona = {"nombre": "Don Beto", "negocio": "abarrotes"}
print(f"{persona['nombre']} tiene una tienda de {persona['negocio'].upper()}")
```

In [ ]:
MI_PREDICCION = ""  #@param {type:"string"}
predecir("""
persona = {"nombre": "Don Beto", "negocio": "abarrotes"}
print(f"{persona['nombre']} tiene una tienda de {persona['negocio'].upper()}")
""", MI_PREDICCION)

### 🧠 Del mapa al enunciado del problema
Después de empatizar, se **define el problema** desde el punto de vista de la persona, y se convierte en una pregunta que abre ideas: **“¿Cómo podríamos…?”** *(Stanford d.school, herramienta “How Might We”.)*

**Plantilla del enunciado:**
> **Hoy** *[persona]* **resuelve esto así:** *[cómo lo hace hoy]*, **y eso le causa:** *[consecuencia]*.
> **¿Cómo podríamos** *[ayudarle a lograr su necesidad]*?

**Ejemplo (Don Beto):**
> Hoy Don Beto resuelve esto así: anota los fiados en un cuaderno y cuenta la caja al cerrar, y eso le causa: no saber cuánto ganó ni quién le debe.
> ¿Cómo podríamos ayudarle a saber al final del día cuánto vendió y cuánto le deben?

### 🧩 Reto R1.2 ERP · El enunciado de TU problema

In [ ]:
HOY_LO_RESUELVE_ASI = ""  #@param {type:"string"}
Y_ESO_LE_CAUSA = ""  #@param {type:"string"}
COMO_PODRIAMOS = ""  #@param {type:"string"}

persona = leer_json('mapa_empatia.json', {}).get('persona', 'la persona')
enunciado = {
    "persona": persona,
    "hoy": HOY_LO_RESUELVE_ASI.strip(),
    "le_causa": Y_ESO_LE_CAUSA.strip(),
    "como_podriamos": COMO_PODRIAMOS.strip().rstrip('?').lstrip('¿'),
}
texto = (f"Hoy {persona} resuelve esto así: {enunciado['hoy']}, y eso le causa: {enunciado['le_causa']}.\n"
         f"¿Cómo podríamos {enunciado['como_podriamos']}?")
enunciado["texto"] = texto
print(texto)
guardar_json('enunciado.json', enunciado)

In [ ]:
_en = lambda: leer_json('enunciado.json', {})
autoverificar('Enunciado del problema', [
    ('Describe cómo lo resuelve hoy', lambda: len(_en().get('hoy', '')) >= 15, 'Describe la forma actual con detalle: libreta, memoria, WhatsApp…'),
    ('Describe la consecuencia', lambda: len(_en().get('le_causa', '')) >= 15, '¿Qué pierde la persona? Tiempo, dinero, clientes, tranquilidad…'),
    ('Tiene la pregunta “¿Cómo podríamos…?”', lambda: len(_en().get('como_podriamos', '')) >= 15, 'Empieza con un verbo: "ayudarle a…", "lograr que…".'),
    ('La pregunta no propone ya la solución', lambda: len(_en().get('como_podriamos', '')) >= 15 and not re.search(r'\b(app|aplicaci[oó]n|sistema|software|p[aá]gina)\b', _en().get('como_podriamos', ''), re.I), 'Todavía no hablamos de la app: enfócate en lo que la persona quiere lograr.'),
])

### 🔎 Técnicas de indagación
| Técnica | Cuándo usarla | En tu proyecto |
|---|---|---|
| 🎙️ **Entrevista a profundidad** | Para entender experiencias y razones, una persona a la vez | Con el dueño o encargado del negocio |
| 📋 **Encuesta** | Para conocer la opinión de muchas personas con preguntas iguales | Con clientes del negocio (Google Forms, sesión 5) |
| 👀 **Observación en contexto** | Para ver lo que la gente **hace** (no solo lo que dice) en su lugar real | Observa 10 minutos cómo se atiende y se cobra |

*(GOV.UK Service Manual, User research: entrevistas a profundidad y observación contextual.)*

**Antes de entrevistar:** explica para qué es la entrevista y **pide permiso** (consentimiento) para tomar notas, y cuida los datos personales de quien te ayuda. *(GOV.UK.)*

### ✍️ Cómo escribir preguntas que no sesguen
Se trata de **explorar** las necesidades de la persona, **no de confirmar lo que ya supones**. Pregunta por experiencias reales y deja que responda con sus palabras. *(GOV.UK, User research.)*

| ❌ Evita | Por qué | ✅ Mejor |
|---|---|---|
| ¿Te gustaría una app para tus ventas? | Hipotética, cerrada y sugiere la respuesta | ¿Cómo registras hoy lo que vendes? |
| ¿Verdad que es difícil llevar el inventario? | Sesgada: empuja a decir que sí | ¿Qué haces cuando se te acaba un producto? |
| ¿Cuánto vendes y quién te ayuda? | Doble: dos preguntas en una | Pregunta una cosa a la vez |
| ¿Usas Excel? | Cerrada (sí/no) | Cuéntame qué herramientas usas para llevar tus cuentas. |

In [ ]:
def revisar_pregunta(pregunta, encuesta=False):
    p = pregunta.strip()
    problemas = []
    if not (p.startswith('¿') and p.endswith('?')) and not re.match(r'(cu[eé]ntame|describe|platica|explica)', p, re.I):
        problemas.append('Escríbela entre ¿ ? o empieza con "Cuéntame…" / "Describe…".')
    if re.search(r'\b(verdad|no crees|no te parece|no es cierto|¿no\?)', p, re.I):
        problemas.append('Sesgada: sugiere la respuesta.')
    if re.search(r'\b(te gustar[ií]a|usar[ií]as|comprar[ií]as|pagar[ií]as|querr[ií]as)\b', p, re.I):
        problemas.append('Hipotética: pregunta por lo que hace HOY, no por lo que haría.')
    if p.count('?') > 1 or re.search(r'\? *y|, y (qu[eé]|c[oó]mo|cu[aá]nt|qui[eé]n|d[oó]nde)', p, re.I):
        problemas.append('Doble: separa en dos preguntas.')
    if not encuesta and re.match(r'¿ *(usas|tienes|has|es|est[aá]s?|te|hay|sabes|puedes|conoces)\b', p, re.I):
        problemas.append('Cerrada (sí/no): ábrela con qué, cómo, cuándo o cuéntame.')
    if re.search(r'\b(app|aplicaci[oó]n|sistema|software)\b', p, re.I):
        problemas.append('Habla de la solución: pregunta por el problema.')
    return problemas


def revisar_preguntas(preguntas, encuesta=False):
    limpias = 0
    for i, p in enumerate(preguntas, 1):
        problemas = revisar_pregunta(p, encuesta)
        print(f"{'✅' if not problemas else '⚠️'} {i}. {p}")
        for problema in problemas:
            print(f"      💡 {problema}")
        limpias += not problemas
    return limpias


revisar_preguntas(["¿Te gustaría una app para tus ventas?", "¿Cómo registras hoy lo que vendes?"])

### 🧩 Reto R1.3 Base · Rescata estas 5 preguntas
Reescribe cada pregunta para que el revisor no marque ninguna advertencia.

In [ ]:
preguntas_malas = [
    "¿Verdad que pierdes mucho tiempo haciendo cuentas?",
    "¿Usarías un sistema para controlar tu inventario?",
    "¿Cuántos clientes atiendes y qué productos vendes más?",
    "¿Tienes problemas con los fiados?",
    "¿No crees que sería mejor cobrar con tarjeta?",
]

preguntas_mejores = [
    "",  # ✏️ 1
    "",  # ✏️ 2
    "",  # ✏️ 3 (puedes separarla en dos preguntas)
    "",  # ✏️ 4
    "",  # ✏️ 5
]

print("ANTES"); revisar_preguntas(preguntas_malas)
print("\nDESPUÉS"); revisar_preguntas([p for p in preguntas_mejores if p.strip()])

In [ ]:
autoverificar('Rescate de preguntas', [
    ('Escribiste 5 preguntas o más', lambda: len([p for p in preguntas_mejores if p.strip()]) >= 5, 'Llena las 5 preguntas.'),
    ('Ninguna tiene advertencias', lambda: len([p for p in preguntas_mejores if p.strip()]) >= 5 and all(not revisar_pregunta(p) for p in preguntas_mejores if p.strip()), 'Lee los 💡 del revisor y corrige.'),
    ('No copiaste las originales', lambda: any(p.strip() for p in preguntas_mejores) and not set(preguntas_mejores) & set(preguntas_malas), 'Reescríbelas con otras palabras.'),
])

### 🧩 Reto R1.3 ERP · Tu guía de entrevista
Escribe **6 preguntas o más** para entrevistar al dueño (real o simulado) de tu negocio. Úsalas para confirmar o corregir tu mapa de empatía.

> 📌 **Tarea para la sesión 6:** haz la entrevista (10 minutos), pide permiso para tomar notas y anota las respuestas en `docs/notas_entrevista.txt` (o en una celda de texto).

In [ ]:
mi_guia = [
    "",  # ✏️ empieza por cómo trabaja hoy
    "",
    "",
    "",
    "",
    "",
]
guia = [p.strip() for p in mi_guia if p.strip()]
limpias = revisar_preguntas(guia)
guardar_json('guia_entrevista.json', guia)

In [ ]:
_g = lambda: leer_json('guia_entrevista.json', [])
autoverificar('Guía de entrevista', [
    ('Tiene 6 preguntas o más', lambda: len(_g()) >= 6, 'Agrega más preguntas sobre ventas, inventario, clientes y tu módulo distintivo.'),
    ('Ninguna tiene advertencias', lambda: bool(_g()) and all(not revisar_pregunta(p) for p in _g()), 'Corrige los 💡 del revisor.'),
    ('Pregunta por el módulo distintivo', lambda: any(any(w in p.lower() for w in re.findall(r'\w{5,}', leer_json('ficha.json', {}).get('modulo_distintivo', '').lower())) for p in _g()), 'Incluye una pregunta sobre tu módulo distintivo (citas, pedidos, fiados…).'),
])

### ⭐ Bonus
Convierte tu guía en una **página HTML imprimible**: un `<ol>` con las preguntas y un espacio `<p>` bajo cada una para anotar la respuesta. Revísala con `revisar_html()` y guárdala con `guardar('docs/guia_entrevista.html', ...)`.

### 🎟️ Ticket de salida · Sesión 4

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(4, APRENDI, ME_BLOQUEO)

---
# 🟦 SESIÓN 5 · Indagación: encuesta en Forms y clasificación de requerimientos
**Reto R1.3 🟢 Base** · Diseñar una encuesta y clasificar requerimientos en funcionales y no funcionales, priorizándolos con MoSCoW.

### 🔮 Predice
```python
reqs = [{"id": "R1", "prioridad": "M"}, {"id": "R2", "prioridad": "C"}, {"id": "R3", "prioridad": "M"}]
print([r["id"] for r in reqs if r["prioridad"] == "M"])
```

In [ ]:
MI_PREDICCION = ""  #@param {type:"string"}
predecir("""
reqs = [{"id": "R1", "prioridad": "M"}, {"id": "R2", "prioridad": "C"}, {"id": "R3", "prioridad": "M"}]
print([r["id"] for r in reqs if r["prioridad"] == "M"])
""", MI_PREDICCION)

### 🧠 Requerimientos funcionales y no funcionales
| Tipo | Responde a… | Ejemplos de un ERP |
|---|---|---|
| **Funcional (F)** | **¿Qué debe hacer** el sistema? Acciones, comportamientos e interacciones | *El sistema debe registrar una venta con sus productos y cantidades.* · *El sistema debe mostrar los productos con stock bajo.* |
| **No funcional (NF)** | **¿Qué tan bien** debe hacerlo? Rendimiento, seguridad, usabilidad, disponibilidad | *La búsqueda de un producto debe responder en menos de 2 segundos.* · *Las contraseñas deben guardarse cifradas.* |

*(Visure Solutions, “Requisitos funcionales vs. no funcionales”; IBM, “¿Qué es la gestión de requisitos?”; Wikipedia, “Requisito (sistemas)”.)*

### 🎯 Priorización MoSCoW
| Letra | Significado | ¿Entra al MVP? |
|---|---|---|
| **M** · *Must have* | **Imprescindible**: sin esto, el proyecto no funciona | ✅ Sí |
| **S** · *Should have* | **Debería tener**: importante, pero no indispensable | Si alcanza el tiempo |
| **C** · *Could have* | **Podría tener**: deseable; si no se hace, no afecta lo demás | Después |
| **W** · *Won't have* | **No tendrá** (en esta versión) | ❌ No |

MoSCoW ayuda a definir el **producto mínimo viable (MVP)**: lo más pequeño que ya resuelve el problema. *(Atlassian, “Seis marcos de priorización de productos”.)*

### 📋 Encuesta en Google Forms (15 min)
Crea en **Google Forms** una encuesta de **5 preguntas** para los **clientes** de tu negocio. Combina tipos: opción múltiple, casillas, escala lineal y una abierta. En una encuesta las preguntas cerradas sí se permiten, pero **no deben ser sesgadas ni dobles**.

Guarda el formulario en la carpeta de tu proyecto y pega abajo el enlace para responder (*Enviar → 🔗*).

In [ ]:
LINK_FORMS = ""  #@param {type:"string"}

preguntas_forms = [
    "",  # ✏️ copia aquí tus 5 preguntas de Forms
    "",
    "",
    "",
    "",
]
encuesta = {"link": LINK_FORMS.strip(), "preguntas": [p.strip() for p in preguntas_forms if p.strip()]}
revisar_preguntas(encuesta["preguntas"], encuesta=True)
guardar_json('encuesta.json', encuesta)

### 🧩 Reto R1.3 Base · Clasifica los requerimientos de la papelería
1. Escribe en cada requerimiento su **tipo** (`"F"` o `"NF"`) y su **prioridad** (`"M"`, `"S"`, `"C"` o `"W"`).
2. Programa `filtrar_por_prioridad(reqs, prioridad)` y `contar_por_tipo(reqs)`.

In [ ]:
requerimientos_papeleria = [
    {"id": "R01", "texto": "El sistema debe registrar los productos con nombre, precio y existencia.", "tipo": "", "prioridad": ""},
    {"id": "R02", "texto": "El sistema debe registrar una venta con varios productos.", "tipo": "", "prioridad": ""},
    {"id": "R03", "texto": "El sistema debe descontar del inventario lo vendido.", "tipo": "", "prioridad": ""},
    {"id": "R04", "texto": "El sistema debe mostrar el total vendido en el día.", "tipo": "", "prioridad": ""},
    {"id": "R05", "texto": "Las pantallas deben poder usarse desde un celular de 360 px de ancho.", "tipo": "", "prioridad": ""},
    {"id": "R06", "texto": "El sistema debe avisar cuando un producto llegue a su existencia mínima.", "tipo": "", "prioridad": ""},
    {"id": "R07", "texto": "La búsqueda de un producto debe mostrar resultados en menos de 2 segundos.", "tipo": "", "prioridad": ""},
    {"id": "R08", "texto": "El sistema debe registrar los pedidos de impresión con su estado.", "tipo": "", "prioridad": ""},
    {"id": "R09", "texto": "Las contraseñas deben guardarse cifradas, nunca como texto visible.", "tipo": "", "prioridad": ""},
    {"id": "R10", "texto": "El sistema debe enviar promociones por WhatsApp a los clientes.", "tipo": "", "prioridad": ""},
    {"id": "R11", "texto": "Los textos deben tener un contraste de color de 4.5:1 o más.", "tipo": "", "prioridad": ""},
    {"id": "R12", "texto": "El sistema debe generar facturas electrónicas ante el SAT.", "tipo": "", "prioridad": ""},
]


def filtrar_por_prioridad(reqs, prioridad):
    pass  # ✏️ TODO: lista de los requerimientos con esa prioridad


def contar_por_tipo(reqs):
    pass  # ✏️ TODO: diccionario {"F": n, "NF": m}

In [ ]:
_TIPOS = json.loads(base64.b64decode("WyJGIiwgIkYiLCAiRiIsICJGIiwgIk5GIiwgIkYiLCAiTkYiLCAiRiIsICJORiIsICJGIiwgIk5GIiwgIkYiXQ=="))
_prueba = [{"id": "A", "tipo": "F", "prioridad": "M"}, {"id": "B", "tipo": "NF", "prioridad": "S"}, {"id": "C", "tipo": "F", "prioridad": "M"}]

autoverificar('Clasificación de requerimientos', [
    ('Pegaste el enlace de tu encuesta', lambda: re.match(r'https://(forms\.gle/|docs\.google\.com/forms/)', leer_json('encuesta.json', {}).get('link', '')), 'Copia el enlace desde Enviar → 🔗 en Google Forms.'),
    ('La encuesta tiene 5 preguntas sin sesgo', lambda: len(leer_json('encuesta.json', {}).get('preguntas', [])) >= 5 and all(not revisar_pregunta(p, encuesta=True) for p in leer_json('encuesta.json', {}).get('preguntas', [])), 'Copia tus 5 preguntas y corrige los 💡.'),
    ('Todos tienen tipo y prioridad válidos', lambda: all(r["tipo"] in ("F", "NF") and r["prioridad"] in ("M", "S", "C", "W") for r in requerimientos_papeleria), 'Usa solo "F"/"NF" y "M"/"S"/"C"/"W".'),
    ('Al menos 9 de 12 tipos correctos', lambda: sum(r["tipo"] == t for r, t in zip(requerimientos_papeleria, _TIPOS)) >= 9, '¿Responde a QUÉ hace el sistema (F) o a QUÉ TAN BIEN lo hace (NF)?'),
    ('Hay al menos un W', lambda: any(r["prioridad"] == "W" for r in requerimientos_papeleria), 'Algo debe quedar fuera de esta versión: ¿facturación ante el SAT?'),
    ('filtrar_por_prioridad() funciona', lambda: [r["id"] for r in filtrar_por_prioridad(_prueba, "M")] == ["A", "C"], 'Recorre la lista y conserva los que tengan esa prioridad.'),
    ('contar_por_tipo() funciona', lambda: contar_por_tipo(_prueba) == {"F": 2, "NF": 1}, 'Empieza con {"F": 0, "NF": 0} y suma 1 según r["tipo"].'),
], meta='75 % o más')

### ⭐ Bonus
Escribe `ordenar_por_prioridad(reqs)` que devuelva la lista ordenada M → S → C → W. **Pista:** `sorted(reqs, key=...)` con un diccionario `{"M": 0, "S": 1, "C": 2, "W": 3}`.

### 🎟️ Ticket de salida · Sesión 5

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(5, APRENDI, ME_BLOQUEO)

---
# 🟦 SESIÓN 6 · Los requerimientos de TU negocio, bien escritos
**Reto R1.3 🔵 ERP** · Redactar 8 o más requerimientos del negocio, claros y verificables, priorizados con MoSCoW.

### 🔮 Predice
```python
prioridad = {"M": 0, "S": 1, "C": 2, "W": 3}
reqs = ["C", "M", "W", "S", "M"]
print(sorted(reqs, key=lambda p: prioridad[p]))
```

In [ ]:
MI_PREDICCION = ""  #@param {type:"string"}
predecir("""
prioridad = {"M": 0, "S": 1, "C": 2, "W": 3}
reqs = ["C", "M", "W", "S", "M"]
print(sorted(reqs, key=lambda p: prioridad[p]))
""", MI_PREDICCION)

### 🧠 ¿Cómo se ve un buen requerimiento?
La norma **ISO/IEC/IEEE 29148** pide que cada requerimiento sea:

| Característica | Qué significa | Pregunta para revisarlo |
|---|---|---|
| **Necesario** | Si lo quitas, falta algo esencial | ¿Qué pasa si no lo hacemos? |
| **Apropiado** | No impone detalles de diseño innecesarios | ¿Dice *qué* sin decir *con qué botón*? |
| **No ambiguo** | Solo se puede entender de una manera | ¿Dos personas lo entenderían igual? |
| **Completo** | Se entiende sin información extra | ¿Le falta algún dato? |
| **Singular** | Pide **una sola** cosa | ¿Tiene un “y” que esconde dos requisitos? |
| **Factible** | Se puede hacer con el tiempo y los recursos que hay | ¿Lo podemos programar este semestre? |
| **Verificable** | Se puede comprobar con una prueba | ¿Cómo sabremos que ya se cumplió? |
| **Correcto** | Representa la necesidad real de la persona | ¿Salió de la entrevista o la encuesta? |
| **Conforme** | Sigue la plantilla acordada | ¿Empieza con *“El sistema debe…”*? |

**Errores comunes** *(Altium, “Errores comunes en la gestión de requisitos”)*:
- **Requisitos vagos:** palabras como *rápido*, *fácil* o *amigable* se interpretan distinto. Usa números: *“en menos de 2 segundos”*.
- ***Gold-plating*:** convertir lo deseable en obligatorio infla el proyecto sin aportar valor. Usa bien la **W** de MoSCoW.
- **Sobreespecificación:** poner restricciones que nadie pidió.

**Plantilla:** `El sistema debe <acción> <qué> [<condición o medida>].`

In [ ]:
PALABRAS_VAGAS = r'\b(r[aá]pid[oa]s?|f[aá]cil(es|mente)?|amigable|bonit[oa]|intuitiv[oa]|eficiente|moderno|adecuad[oa]|etc|mejor|optimizad[oa]|sencill[oa])\b'


def revisar_requisito(req):
    texto = req.get("texto", "").strip()
    problemas = []
    if not re.match(r'(el sistema|la aplicaci[oó]n|el erp) debe', texto, re.I) and not re.match(r'(las?|los?|solo|el) .+ debe', texto, re.I):
        problemas.append('Conforme: usa la plantilla "El sistema debe…".')
    if re.search(PALABRAS_VAGAS, texto, re.I):
        problemas.append(f"Ambiguo: cambia '{re.search(PALABRAS_VAGAS, texto, re.I).group()}' por algo medible.")
    if re.search(r'\b(y|e) (tambi[eé]n )?(debe|permitir|registrar|mostrar|enviar|calcular|generar)', texto, re.I):
        problemas.append('Singular: parece que pide dos cosas; sepáralo en dos requerimientos.')
    if req.get("tipo") == "NF" and not re.search(r'\d', texto):
        problemas.append('Verificable: un no funcional necesita una medida (segundos, px, %, intentos…).')
    if len(texto) < 25:
        problemas.append('Completo: le faltan detalles.')
    if req.get("tipo") not in ("F", "NF"):
        problemas.append('Tipo: escribe "F" o "NF".')
    if req.get("prioridad") not in ("M", "S", "C", "W"):
        problemas.append('Prioridad: escribe "M", "S", "C" o "W".')
    return problemas


def revisar_requerimientos(reqs):
    buenos = 0
    for r in reqs:
        problemas = revisar_requisito(r)
        buenos += not problemas
        print(f"{'✅' if not problemas else '⚠️'} {r.get('id', '?')} [{r.get('tipo', '?')}/{r.get('prioridad', '?')}] {r.get('texto', '')}")
        for p in problemas:
            print(f"      💡 {p}")
    return buenos


revisar_requerimientos([
    {"id": "X1", "texto": "El sistema debe ser rápido y fácil de usar.", "tipo": "NF", "prioridad": "M"},
    {"id": "X2", "texto": "La lista de productos debe cargar en menos de 2 segundos con 500 productos.", "tipo": "NF", "prioridad": "S"},
])

### 🧩 Reto R1.3 ERP · 8 o más requerimientos de TU negocio
Cada requerimiento lleva su **fuente**: de dónde salió (`"entrevista"`, `"encuesta"`, `"observación"` o `"supuesto"`). A eso se le llama **trazabilidad**.

Incluye el **núcleo común** (productos, clientes, ventas, inventario, reporte del día), tu **módulo distintivo**, **2 o más no funcionales** y **1 o más W**.

In [ ]:
mis_requerimientos = [
    {"id": "R01", "texto": "El sistema debe ", "tipo": "F", "prioridad": "M", "fuente": "entrevista"},
    {"id": "R02", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    {"id": "R03", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    {"id": "R04", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    {"id": "R05", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    {"id": "R06", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    {"id": "R07", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    {"id": "R08", "texto": "", "tipo": "", "prioridad": "", "fuente": ""},
    # ✏️ agrega más si los necesitas
]

reqs = [r for r in mis_requerimientos if len(r["texto"].strip()) > len("El sistema debe ")]
buenos = revisar_requerimientos(reqs)
guardar_json('requerimientos.json', reqs)

In [ ]:
_r = lambda: leer_json('requerimientos.json', [])
autoverificar('Requerimientos de mi negocio', [
    ('Tienes 8 o más requerimientos', lambda: len(_r()) >= 8, 'Revisa tu mapa de empatía: cada necesidad y frustración sugiere un requerimiento.'),
    ('Hay 2 o más no funcionales', lambda: sum(r["tipo"] == "NF" for r in _r()) >= 2, 'Piensa en el celular (360 px), la velocidad, la seguridad o el contraste.'),
    ('Hay al menos un W (fuera de esta versión)', lambda: any(r["prioridad"] == "W" for r in _r()), '¿Qué pidió el dueño que NO cabe en el semestre?'),
    ('Los M no pasan del 60 % (sin gold-plating)', lambda: _r() and sum(r["prioridad"] == "M" for r in _r()) / len(_r()) <= 0.6, 'Si todo es imprescindible, nada lo es: baja algunos a S o C.'),
    ('Todos tienen fuente', lambda: bool(_r()) and all(r.get("fuente") in ("entrevista", "encuesta", "observación", "supuesto") for r in _r()), 'Fuente: "entrevista", "encuesta", "observación" o "supuesto".'),
    ('Incluyen el módulo distintivo', lambda: any(any(w in r["texto"].lower() for w in re.findall(r'\w{5,}', leer_json('ficha.json', {}).get('modulo_distintivo', '').lower())) for r in _r()), 'Agrega al menos un requerimiento de tu módulo distintivo.'),
    ('75 % o más pasan el revisor', lambda: _r() and sum(not revisar_requisito(r) for r in _r()) / len(_r()) >= 0.75, 'Corrige los 💡 del revisor.'),
], meta='75 % o más')

### ⭐ Bonus
Exporta tus requerimientos a **CSV** (`datos/requerimientos.csv`) con `csv.DictWriter` y ábrelo en Google Sheets: agrega un filtro por prioridad. **Pista:** `os.path.join(RUTA_PROYECTO, 'datos', 'requerimientos.csv')`.

### 🎟️ Ticket de salida · Sesión 6

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(6, APRENDI, ME_BLOQUEO)

---
# 🟦 SESIÓN 7 · Historias de usuario, alcance MVP y documento de requerimientos
**Reto R1.4 🟢 Base + 🔵 ERP · Entrega 10 %** · Convertir los requerimientos en historias de usuario con criterios de aceptación, definir el MVP y entregar el documento de requerimientos.

### 🔮 Predice
```python
historia = "Como cajero, quiero registrar una venta para cobrar rápido"
partes = historia.split(", quiero ")
print(partes[0].replace("Como ", ""), "|", len(partes))
```

In [ ]:
MI_PREDICCION = ""  #@param {type:"string"}
predecir("""
historia = "Como cajero, quiero registrar una venta para cobrar rápido"
partes = historia.split(", quiero ")
print(partes[0].replace("Como ", ""), "|", len(partes))
""", MI_PREDICCION)

### 🧠 Historias de usuario
Una historia de usuario describe una funcionalidad **desde el punto de vista de quien la usa**, en lenguaje no técnico:

> **Como** *[perfil]*, **quiero** *[objetivo]* **para** *[beneficio]*.

- **Como…**: ¿para quién? Un perfil real, con el que empatizaste.
- **Quiero…**: su **intención**, no la interfaz. Si describes un botón o una pantalla, es un error.
- **Para…**: el beneficio o el problema de fondo que resuelve.

Cada historia lleva **criterios de aceptación**: las condiciones que dicen **cuándo está terminada**. Las **3 C** son *Card* (la tarjeta con la historia), *Conversation* (la plática que aclara detalles) y *Confirmation* (los criterios de aceptación). *(Atlassian, “Historias de usuario con ejemplos y plantilla”.)*

### ✅ INVEST: una buena historia es…
| Letra | Característica | Significado |
|---|---|---|
| **I** | Independiente | Se puede planear y construir sin depender de otra |
| **N** | Negociable | Es un punto de partida para conversar, no un contrato cerrado |
| **V** | Valiosa | Aporta valor al usuario o al negocio |
| **E** | Estimable | El equipo puede calcular cuánto esfuerzo lleva |
| **S** | Pequeña (*Small*) | Se termina en pocos días |
| **T** | Comprobable (*Testable*) | Se puede verificar con pruebas |

*(Scrum Manager, “El método INVEST y las historias de usuario”, a partir de Bill Wake.)*

**Ejemplo:**
> **Como** dueño de la papelería, **quiero** ver qué productos están por agotarse **para** resurtirlos antes de quedarme sin venta.
> - ✔️ Se muestran los productos con existencia menor o igual a su mínimo.
> - ✔️ La lista se actualiza después de cada venta.
> - ✔️ Si no hay productos por agotarse, aparece “Sin alertas”.

In [ ]:
PATRON_HISTORIA = r'^como (?P<perfil>.+?),? quiero (?P<objetivo>.+?),? para (?P<beneficio>.+)$'
PALABRAS_DE_INTERFAZ = r'\b(bot[oó]n|pantalla|clic|click|men[uú]|pesta[nñ]a|formulario|ventana|icono)\b'


def revisar_historia(h):
    problemas = []
    m = re.match(PATRON_HISTORIA, h.get("historia", "").strip(), re.I)
    if not m:
        problemas.append('Formato: "Como <perfil>, quiero <objetivo> para <beneficio>".')
    else:
        if re.search(PALABRAS_DE_INTERFAZ, m.group('objetivo'), re.I):
            problemas.append('Quiero: describe la intención, no la interfaz (botón, pantalla, clic…).')
        if len(m.group('beneficio')) < 10:
            problemas.append('Para: explica mejor el beneficio.')
    criterios = [c for c in h.get("criterios", []) if c.strip()]
    if len(criterios) < 2:
        problemas.append('Criterios: escribe 2 o más condiciones verificables.')
    if any(re.search(PALABRAS_VAGAS, c, re.I) for c in criterios):
        problemas.append('Criterios: hay palabras vagas; usa condiciones que se puedan comprobar.')
    if not h.get("requerimientos"):
        problemas.append('Trazabilidad: indica qué requerimientos cubre (por ejemplo ["R03"]).')
    return problemas


def revisar_historias(historias):
    buenas = 0
    for i, h in enumerate(historias, 1):
        problemas = revisar_historia(h)
        buenas += not problemas
        print(f"{'✅' if not problemas else '⚠️'} H{i}. {h.get('historia', '')}")
        for c in h.get("criterios", []):
            if c.strip():
                print(f"      ✔️ {c}")
        for p in problemas:
            print(f"      💡 {p}")
    return buenas

### 🧩 Reto R1.4 Base · Rescata estas 3 historias

In [ ]:
historias_rescate = [
    {"historia": "Quiero un botón verde para guardar ventas", "criterios": [], "requerimientos": ["R02"]},
    {"historia": "Como usuario, quiero que el sistema sea rápido para que sea mejor", "criterios": ["Que sea rápido"], "requerimientos": ["R07"]},
    {"historia": "Como dueño, quiero hacer clic en la pantalla de reportes", "criterios": ["Se ve el reporte"], "requerimientos": ["R04"]},
]

# ✏️ Reescribe las 3 historias en esta lista
historias_corregidas = [
    {"historia": "", "criterios": ["", ""], "requerimientos": ["R02"]},
    {"historia": "", "criterios": ["", ""], "requerimientos": ["R07"]},
    {"historia": "", "criterios": ["", ""], "requerimientos": ["R04"]},
]

print("ANTES"); revisar_historias(historias_rescate)
print("\nDESPUÉS"); revisar_historias(historias_corregidas)

In [ ]:
autoverificar('Rescate de historias', [
    ('Las 3 historias pasan el revisor', lambda: all(not revisar_historia(h) for h in historias_corregidas), 'Lee los 💡 de cada historia.'),
], meta='3 de 3')

### 🧩 Reto R1.4 ERP · 6 o más historias de TU negocio + alcance MVP
Escribe tus historias a partir de tus requerimientos **M** y **S**, y conecta cada una con los requerimientos que cubre.

In [ ]:
mis_historias = [
    {"historia": "Como ", "criterios": ["", ""], "requerimientos": [""]},
    {"historia": "", "criterios": ["", ""], "requerimientos": [""]},
    {"historia": "", "criterios": ["", ""], "requerimientos": [""]},
    {"historia": "", "criterios": ["", ""], "requerimientos": [""]},
    {"historia": "", "criterios": ["", ""], "requerimientos": [""]},
    {"historia": "", "criterios": ["", ""], "requerimientos": [""]},
    # ✏️ agrega más si las necesitas
]

historias = [{**h, "criterios": [c for c in h["criterios"] if c.strip()],
              "requerimientos": [r for r in h["requerimientos"] if r.strip()]}
             for h in mis_historias if len(h["historia"].strip()) > len("Como ")]
revisar_historias(historias)
guardar_json('historias.json', historias)

### 📦 Alcance mínimo (MVP)
Marca qué partes del núcleo común entran a tu primera versión. Lo que priorizaste como **W** queda automáticamente **fuera del alcance**.

In [ ]:
PRODUCTOS_O_SERVICIOS = True  #@param {type:"boolean"}
CLIENTES = True  #@param {type:"boolean"}
VENTAS = True  #@param {type:"boolean"}
INVENTARIO_CON_ALERTA = True  #@param {type:"boolean"}
REPORTE_DEL_DIA = True  #@param {type:"boolean"}

ficha = leer_json('ficha.json', {})
reqs = leer_json('requerimientos.json', [])
mvp = {
    "nucleo": {"Productos o servicios": PRODUCTOS_O_SERVICIOS, "Clientes": CLIENTES, "Ventas": VENTAS,
               "Inventario con alerta de stock bajo": INVENTARIO_CON_ALERTA, "Reporte del día": REPORTE_DEL_DIA},
    "modulo_distintivo": ficha.get("modulo_distintivo", ""),
    "dentro": [r["id"] for r in reqs if r["prioridad"] in ("M", "S")],
    "despues": [r["id"] for r in reqs if r["prioridad"] == "C"],
    "fuera": [f'{r["id"]}: {r["texto"]}' for r in reqs if r["prioridad"] == "W"],
}
guardar_json('mvp.json', mvp)
print("✅ Dentro del MVP:", ", ".join(mvp["dentro"]))
print("⏳ Después (C):", ", ".join(mvp["despues"]) or "—")
print("❌ Fuera de esta versión (W):"); [print("   •", f) for f in mvp["fuera"]]

### 📄 Genera tu documento de requerimientos
Esta celda reúne **todo lo de la semana** (ficha, mapa de empatía, enunciado, guía de entrevista, encuesta, requerimientos, historias, MVP y bitácora de IA) en un solo documento.

**Después:**
1. En Drive, abre `docs/Documento_requerimientos.html` con **clic derecho → Abrir con → Documentos de Google**.
2. En el Doc, agrega el **resumen de las respuestas de tu encuesta** (gráficas de Forms) y las **notas de tu entrevista**.
3. Revisa la redacción y guárdalo en la carpeta `docs/` de tu proyecto.

In [ ]:
def generar_documento():
    e = lambda t: html_lib.escape(str(t))
    ficha = leer_json('ficha.json', {})
    mapa = leer_json('mapa_empatia.json', {})
    enunciado = leer_json('enunciado.json', {})
    guia = leer_json('guia_entrevista.json', [])
    encuesta = leer_json('encuesta.json', {})
    reqs = leer_json('requerimientos.json', [])
    historias = leer_json('historias.json', [])
    mvp = leer_json('mvp.json', {})
    ruta_ia = os.path.join(RUTA_PROYECTO, 'bitacora', 'bitacora_ia.csv')
    usos_ia = list(csv.DictReader(open(ruta_ia, encoding='utf-8'))) if os.path.exists(ruta_ia) else []
    nombres = {"dice": "Dice", "hace": "Hace", "le_frustra": "Le frustra", "necesita": "Necesita", "piensa": "Piensa", "siente": "Siente"}

    partes = [f"<h1>Documento de requerimientos · {e(ficha.get('nombre_erp', ''))}</h1>",
              f"<p><b>Micronegocio:</b> {e(ficha.get('negocio', ''))}<br><b>Módulo distintivo:</b> {e(ficha.get('modulo_distintivo', ''))}"
              f"<br><b>Integrantes:</b> {e(', '.join(ficha.get('integrantes', [])))}</p>",
              "<h2>1. La persona y su problema</h2>",
              f"<h3>Mapa de empatía: {e(mapa.get('persona', ''))}</h3>"]
    for clave, frases in mapa.get('mapa', {}).items():
        partes.append(f"<p><b>{nombres.get(clave, clave)}:</b></p><ul>{''.join(f'<li>{e(f)}</li>' for f in frases)}</ul>")
    partes += ["<h3>Enunciado del problema</h3>", f"<p>{e(enunciado.get('texto', '')).replace(chr(10), '<br>')}</p>",
               "<h2>2. Indagación</h2>", "<h3>Guía de entrevista</h3>", f"<ol>{''.join(f'<li>{e(p)}</li>' for p in guia)}</ol>",
               "<p><i>Notas de la entrevista: (agrégalas aquí)</i></p>",
               "<h3>Encuesta a clientes (Google Forms)</h3>", f"<p>Enlace: {e(encuesta.get('link', ''))}</p>",
               f"<ol>{''.join(f'<li>{e(p)}</li>' for p in encuesta.get('preguntas', []))}</ol>",
               "<p><i>Resumen de resultados: (agrega aquí las gráficas de Forms)</i></p>",
               "<h2>3. Requerimientos</h2>",
               "<table border='1' cellpadding='4'><tr><th>ID</th><th>Requerimiento</th><th>Tipo</th><th>Prioridad</th><th>Fuente</th></tr>"]
    for r in reqs:
        partes.append(f"<tr><td>{e(r['id'])}</td><td>{e(r['texto'])}</td><td>{e(r['tipo'])}</td><td>{e(r['prioridad'])}</td><td>{e(r.get('fuente', ''))}</td></tr>")
    partes += ["</table>", "<p>F = funcional · NF = no funcional · M = imprescindible · S = debería · C = podría · W = no en esta versión</p>",
               "<h2>4. Historias de usuario</h2>"]
    for i, h in enumerate(historias, 1):
        partes.append(f"<p><b>H{i}.</b> {e(h['historia'])} <i>(cubre: {e(', '.join(h['requerimientos']))})</i></p>"
                      f"<ul>{''.join(f'<li>{e(c)}</li>' for c in h['criterios'])}</ul>")
    partes += ["<h2>5. Alcance mínimo (MVP)</h2>", "<ul>"]
    partes += [f"<li>{'✅' if v else '❌'} {e(k)}</li>" for k, v in mvp.get('nucleo', {}).items()]
    partes += [f"<li>✅ Módulo distintivo: {e(mvp.get('modulo_distintivo', ''))}</li>", "</ul>",
               f"<p><b>Requerimientos dentro del MVP:</b> {e(', '.join(mvp.get('dentro', [])))}</p>",
               f"<p><b>Para después (C):</b> {e(', '.join(mvp.get('despues', [])) or '—')}</p>",
               "<p><b>Fuera de esta versión (W):</b></p>", f"<ul>{''.join(f'<li>{e(f)}</li>' for f in mvp.get('fuera', []))}</ul>",
               "<h2>6. Bitácora de uso de IA</h2>"]
    if usos_ia:
        partes.append("<table border='1' cellpadding='4'><tr><th>Fecha</th><th>Pregunta</th><th>Qué acepté</th><th>Cómo verifiqué</th></tr>")
        partes += [f"<tr><td>{e(u['fecha'])}</td><td>{e(u['pregunta'])}</td><td>{e(u['que_acepte'])}</td><td>{e(u['como_verifique'])}</td></tr>" for u in usos_ia]
        partes.append("</table>")
    else:
        partes.append("<p>No se usó IA esta semana.</p>")

    documento = ("<!DOCTYPE html><html lang='es'><head><meta charset='UTF-8'><title>Documento de requerimientos</title></head>"
                 "<body style='font-family: Arial, sans-serif'>" + "\n".join(partes) + "</body></html>")
    guardar('docs/Documento_requerimientos.html', documento)
    return documento


documento = generar_documento()
mostrar(documento, alto=600)

### 📋 Lista de cotejo: documento de requerimientos (10 %)
| Elemento | Puntos |
|---|---|
| Portada: nombre del ERP, micronegocio, módulo distintivo e integrantes | 0.5 |
| Mapa de empatía (4 cuadrantes) y enunciado del problema con “¿Cómo podríamos…?” | 1.5 |
| Indagación: guía de entrevista con notas, y encuesta de 5 preguntas con resumen de resultados | 1.5 |
| **8 o más requerimientos** F/NF, priorizados con MoSCoW, con la plantilla y su fuente | 2.5 |
| **6 o más historias de usuario** con criterios de aceptación (INVEST) y requerimientos que cubren | 2.5 |
| Alcance MVP: núcleo común + módulo distintivo + lo que queda fuera | 1.0 |
| Documento en Google Docs, bien redactado y guardado en la carpeta del proyecto | 0.5 |
| **Total** | **10** |

In [ ]:
autoverificar('Semana 1 completa', [
    ('Ficha del proyecto', lambda: bool(leer_json('ficha.json')), 'Sesión 2.'),
    ('Mapa de empatía (4 cuadrantes)', lambda: all(len(leer_json('mapa_empatia.json', {}).get('mapa', {}).get(c, [])) >= 3 for c in ('dice', 'hace', 'le_frustra', 'necesita')), 'Sesión 3.'),
    ('Enunciado del problema', lambda: len(leer_json('enunciado.json', {}).get('hoy', '')) >= 15, 'Sesión 4.'),
    ('Guía de entrevista (6 o más)', lambda: len(leer_json('guia_entrevista.json', [])) >= 6, 'Sesión 4.'),
    ('Encuesta con enlace y 5 preguntas', lambda: bool(leer_json('encuesta.json', {}).get('link')) and len(leer_json('encuesta.json', {}).get('preguntas', [])) >= 5, 'Sesión 5.'),
    ('8 o más requerimientos', lambda: len(leer_json('requerimientos.json', [])) >= 8, 'Sesión 6.'),
    ('6 o más historias que pasan el revisor', lambda: sum(not revisar_historia(h) for h in leer_json('historias.json', [])) >= 6, 'Sesión 7: corrige los 💡.'),
    ('Alcance MVP', lambda: bool(leer_json('mvp.json', {}).get('dentro')), 'Sesión 7.'),
    ('Documento generado', lambda: os.path.exists(os.path.join(RUTA_PROYECTO, 'docs', 'Documento_requerimientos.html')), 'Ejecuta generar_documento().'),
], meta='100 % para entregar')
print()
arbol()

### ⭐ Bonus
Usa **Gemini** para revisar **una** de tus historias con INVEST. Pídele: *“Dime qué letra de INVEST no cumple esta historia y por qué, sin reescribirla”*. Decide tú si le haces caso, y regístralo con `registrar_ia()`.

### 🎟️ Ticket de salida · Sesión 7

In [ ]:
APRENDI = ""  #@param {type:"string"}
ME_BLOQUEO = ""  #@param {type:"string"}
ticket_salida(7, APRENDI, ME_BLOQUEO)

---
### ✅ Autoevaluación de la semana: marca lo que ya cumples
| Criterio | Sesión | ✔️ |
|---|---|---|
| 6 de 8 mini-retos de Python o más | 1 | ☐ |
| Ficha del proyecto y carpeta `ERP_<nombre de tu ERP>` compartida con la pareja | 2 | ☐ |
| Mapa de empatía del negocio (4 cuadrantes, 3 frases o más cada uno) | 3 | ☐ |
| Enunciado del problema compartido en plenaria | 4 | ☐ |
| Guía de entrevista de 6 preguntas sin sesgo y entrevista realizada | 4 · 6 | ☐ |
| Encuesta de 5 preguntas en Google Forms | 5 | ☐ |
| 8 o más requerimientos priorizados con MoSCoW | 6 | ☐ |
| 6 o más historias con criterios de aceptación + MVP | 7 | ☐ |
| Documento de requerimientos en Google Docs | 7 | ☐ |
| 7 tickets de salida registrados | 1–7 | ☐ |

**Siguiente semana:** diseño de pantallas y prototipo navegable (flujo de datos, accesibilidad WCAG, mobile-first y prototipo en Google Slides).

---
## 📚 Documentación y bibliografía
Fuentes del notebook de contexto del curso **«Construye App WEB»** usadas en este cuaderno:

| Fuente | Tema | Sesión |
|---|---|---|
| DGETI. *Programa de estudios de Programación* (2023) | Módulo IV, Submódulo 1: actividad clave «Diseña aplicaciones web» y resultado de aprendizaje | 2 |
| *Planeación didáctica del Submódulo 1: Construye aplicaciones web* | Secuencia de requerimientos, historias de usuario y documento en Docs | Toda la semana |
| Google for Developers. *Colab* | Entorno de programación | 1 |
| Stanford d.school. *Design Tools for Creative Thinking* | Diseño centrado en las personas, empatía, preguntas “How Might We” | 3 · 4 |
| GOV.UK Service Manual. *User research* | Entrevistas a profundidad, observación contextual, consentimiento informado | 4 |
| Visure Solutions. *Requisitos funcionales vs. no funcionales (con ejemplos)* | Requerimientos F y NF | 5 |
| IBM. *¿Qué es la gestión de requisitos?* | Gestión de requisitos; tipos funcional, no funcional y de dominio | 5 |
| Wikipedia. *Requisito (sistemas)* | Definición y características de un buen requisito | 5 · 6 |
| Atlassian. *Seis marcos de priorización de productos y cómo elegir el correcto* | Priorización MoSCoW y MVP | 5 · 6 |
| ISO/IEC/IEEE 29148. *Systems and software engineering: Requirements engineering* | Características de un requisito individual y de un conjunto de requisitos | 6 |
| Altium Learning Hub. *Errores comunes en la gestión de requisitos (y cómo evitarlos)* | Requisitos vagos, *gold-plating*, sobreespecificación | 6 |
| IREB. *CPRE Foundation Level* | Ingeniería de requisitos (consulta) | 6 |
| Atlassian. *Historias de usuario con ejemplos y plantilla* | Formato *Como… quiero… para…*, criterios de aceptación, las 3 C | 7 |
| Scrum Manager. *El método INVEST y las historias de usuario* | INVEST (Bill Wake) | 7 |
| Gaona Toro, X. *Impacto de la Inteligencia Artificial en la Gestión de Requerimientos de Software* | Uso de IA en requerimientos: beneficios y desafíos | 7 (Bonus) |
| Google Workspace. *Google Docs Editors Help* · *Google Sheets training and help* | Documento de requerimientos y exportación a Sheets | 6 · 7 |
| W3C WAI. *Sumario de WCAG 2* | Contraste 4.5:1 en los requerimientos no funcionales | 5 · 6 |